# Carga da base StagingArea no SQL Server

Cria a base `StagingArea` (11 tabelas) a partir do script da formadora.
Só precisa de correr **uma vez** por computador. Volta a criar as tabelas do zero, por isso é seguro repetir.

Os indicadores financeiros não estão aqui: carregam-se no MongoDB com `carga_mongodb.mongodb.js`.

In [2]:
import re
from pathlib import Path
import pyodbc

SQL_SERVER = "localhost"                                   # instância MSSQLSERVER
FICHEIRO   = Path("base_dados/StagingArea_formacao_dados.sql")

# Escolhe automaticamente o driver ODBC mais recente instalado
drivers = sorted(d for d in pyodbc.drivers() if re.match(r"ODBC Driver \d+ for SQL Server", d))
assert drivers, "Nenhum ODBC Driver for SQL Server encontrado. Instala o ODBC Driver 18."
ODBC_DRIVER = drivers[-1]

assert FICHEIRO.exists(), f"Ficheiro não encontrado: {FICHEIRO.resolve()}"
print("Driver:", ODBC_DRIVER)
print("Script:", FICHEIRO.resolve())

Driver: ODBC Driver 18 for SQL Server
Script: E:\AIDAPT\Python Stuff\Projeto ML\Template_CRISP-DM_v2.0_Formandos\Setup\Base_Dados\StagingArea_formacao_dados.sql


In [3]:
script = FICHEIRO.read_text(encoding="utf-8-sig")

lotes = [l.strip()
         for l in re.split(r"^\s*GO\s*$", script, flags=re.MULTILINE | re.IGNORECASE)
         if l.strip()]

print(f"{len(lotes)} lotes para executar")

119 lotes para executar


In [4]:
conn_str = (f"DRIVER={{{ODBC_DRIVER}}};SERVER={SQL_SERVER};DATABASE=master;"
            "Trusted_Connection=yes;TrustServerCertificate=yes;")

con = pyodbc.connect(conn_str, autocommit=True)   # autocommit: o CREATE DATABASE não corre numa transação
cur = con.cursor()

for i, lote in enumerate(lotes, 1):
    cur.execute(lote)
    while cur.nextset():
        pass
    if i % 10 == 0 or i == len(lotes):
        print(f"{i}/{len(lotes)} lotes")

print("Carga concluída")

10/119 lotes
20/119 lotes
30/119 lotes
40/119 lotes
50/119 lotes
60/119 lotes
70/119 lotes
80/119 lotes
90/119 lotes
100/119 lotes
110/119 lotes
119/119 lotes
Carga concluída
